# Transformer from scratch; Sentiment Classifier

An encoder-only transformer, trained from scratch to classify IMDB movie reviews as positive or negative.


## 1. Data — IMDB movie reviews (50k, balanced positive/negative)

In [1]:
import csv, re, time, random, requests, torch, torch.nn as nn
from collections import Counter

torch.manual_seed(42)
random.seed(42)

url = "https://raw.githubusercontent.com/Ankit152/IMDB-sentiment-analysis/master/IMDB-Dataset.csv"
csv_text = requests.get(url).text
rows = list(csv.DictReader(csv_text.splitlines()))
random.shuffle(rows)
print(f"{len(rows):,} reviews | {sum(1 for r in rows if r['sentiment']=='positive'):,} positive / {sum(1 for r in rows if r['sentiment']=='negative'):,} negative")
print(rows[0]['sentiment'], '-', rows[0]['review'][:200])

50,008 reviews | 25,000 positive / 25,000 negative
positive - The movie has a distinct (albeit brutish and rough) humanity for all its borderline depravity - the zippy/lyrical score points up the comic side of their misadventures, and even when they're at their 


In [2]:
n_val = 5000
val_rows, train_rows = rows[:n_val], rows[n_val:]
print(f"train: {len(train_rows):,} | val: {len(val_rows):,}")

train: 45,008 | val: 5,000


## 2. Tokenizer


In [3]:
def clean(text):
    return re.sub(r'<[^>]+>', ' ', text)

def tokenize(text):
    return re.findall(r"[a-z0-9]+|[^\sa-z0-9]", clean(text).lower())

counts = Counter()
for r in train_rows:
    counts.update(tokenize(r['review']))

VOCAB_SIZE = 8000
specials = ["<pad>", "<unk>", "<cls>"]
vocab = specials + [w for w, _ in counts.most_common(VOCAB_SIZE - len(specials))]
stoi = {w: i for i, w in enumerate(vocab)}
PAD, UNK, CLS = stoi["<pad>"], stoi["<unk>"], stoi["<cls>"]
vocab_size = len(vocab)
print("vocab_size:", vocab_size)

vocab_size: 8000


In [4]:
MAX_LEN = 200

def encode(text, max_len=MAX_LEN):
    ids = [stoi.get(w, UNK) for w in tokenize(text)][:max_len]
    ids = [CLS] + ids
    pad_len = (max_len + 1) - len(ids)
    mask = [1] * len(ids) + [0] * pad_len
    ids = ids + [PAD] * pad_len
    return ids, mask

def build_split(rows_subset, n=None):
    subset = rows_subset[:n] if n else rows_subset
    X, M, Y = [], [], []
    for r in subset:
        ids, mask = encode(r['review'])
        X.append(ids); M.append(mask)
        Y.append(1 if r['sentiment'] == 'positive' else 0)
    return torch.tensor(X), torch.tensor(M), torch.tensor(Y)

t0 = time.time()
Xtr, Mtr, Ytr = build_split(train_rows, n=8000)
Xva, Mva, Yva = build_split(val_rows, n=2000)
print(f"encoded in {time.time()-t0:.1f}s | train {len(Xtr)} val {len(Xva)}")

def get_batch(split, batch_size):
    X, M, Y = (Xtr, Mtr, Ytr) if split == 'train' else (Xva, Mva, Yva)
    ix = torch.randint(len(X), (batch_size,))
    return X[ix], M[ix], Y[ix]

encoded in 2.4s | train 8000 val 2000


## 3. Self-attention — bidirectional, with a padding mask instead of a causal mask

Same core formula as any transformer:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

The only mask applied here blocks attention **to padding positions** (so real tokens don't waste attention weight on meaningless `<pad>` filler), there's no restriction on attending to *future* tokens, since the whole review is available at once. This is what makes it bidirectional rather than causal.

In [5]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, embedding_dim, n_heads, dropout=0.1):
        super().__init__()
        assert embedding_dim % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = embedding_dim // n_heads
        self.qkv = nn.Linear(embedding_dim, 3 * embedding_dim)
        self.proj = nn.Linear(embedding_dim, embedding_dim)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

    def forward(self, x, key_padding_mask):
        B, T, C = x.shape
        qkv = self.qkv(x)
        q, k, v = qkv.split(C, dim=2)
        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        attn = (q @ k.transpose(-2, -1)) / (self.head_dim ** 0.5)
        mask = key_padding_mask[:, None, None, :].bool()
        attn = attn.masked_fill(~mask, float('-inf'))
        attn = torch.softmax(attn, dim=-1)
        attn = self.attn_dropout(attn)

        out = attn @ v
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.proj(out))

## 4. Transformer


In [6]:
class TransformerBlock(nn.Module):
    def __init__(self, embedding_dim, n_heads, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(embedding_dim)
        self.attn = MultiHeadSelfAttention(embedding_dim, n_heads, dropout)
        self.ln2 = nn.LayerNorm(embedding_dim)
        self.ff = nn.Sequential(
            nn.Linear(embedding_dim, embedding_dim * 4),
            nn.GELU(),
            nn.Linear(embedding_dim * 4, embedding_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x, mask):
        x = x + self.attn(self.ln1(x), mask)
        x = x + self.ff(self.ln2(x))
        return x

## 5. The classifier


In [7]:
class TransformerClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, max_len, n_blocks, n_heads, n_classes=2, dropout=0.1):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=PAD)
        self.position_embedding = nn.Embedding(max_len, embedding_dim)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([TransformerBlock(embedding_dim, n_heads, dropout) for _ in range(n_blocks)])
        self.ln_f = nn.LayerNorm(embedding_dim)
        self.head = nn.Linear(embedding_dim, n_classes)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, mask, targets=None):
        B, T = idx.shape
        x = self.drop(self.token_embedding(idx) + self.position_embedding(torch.arange(T, device=idx.device)))
        for block in self.blocks:
            x = block(x, mask)
        x = self.ln_f(x)
        cls_repr = x[:, 0, :]
        logits = self.head(cls_repr)
        loss = None
        if targets is not None:
            loss = nn.functional.cross_entropy(logits, targets)
        return logits, loss

## 6. Training

In [8]:
embedding_dim = 64
n_heads = 4
n_blocks = 2
dropout = 0.1
batch_size = 32
max_iters = 1000
eval_interval = 100
lr = 1e-3

model = TransformerClassifier(vocab_size, embedding_dim, MAX_LEN + 1, n_blocks, n_heads, dropout=dropout)
n_params = sum(p.numel() for p in model.parameters())
print(f"Params: {n_params/1e6:.2f}M")

optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.1)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_iters)

best_val_acc = 0
best_state = None
t0 = time.time()
for it in range(max_iters + 1):
    if it % eval_interval == 0 or it == max_iters:
        model.eval()
        with torch.no_grad():
            accs, losses = [], []
            for _ in range(10):
                xv, mv, yv = get_batch('val', 64)
                vlogits, vloss = model(xv, mv, yv)
                accs.append((vlogits.argmax(-1) == yv).float().mean().item())
                losses.append(vloss.item())
            va, vl = sum(accs) / len(accs), sum(losses) / len(losses)
        model.train()
        marker = ''
        if va > best_val_acc:
            best_val_acc = va
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            marker = '  <- best'
        print(f"step {it}: val loss {vl:.4f} val acc {va:.4f} elapsed {time.time()-t0:.0f}s{marker}")
    x, m, y = get_batch('train', batch_size)
    logits, loss = model(x, m, y)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    scheduler.step()

model.load_state_dict(best_state)
print(f"loaded best checkpoint, val acc {best_val_acc:.4f}")

Params: 0.63M
step 0: val loss 0.7167 val acc 0.4625 elapsed 3s  <- best
step 100: val loss 0.5053 val acc 0.7406 elapsed 47s  <- best
step 200: val loss 0.4924 val acc 0.7859 elapsed 89s  <- best
step 300: val loss 0.5334 val acc 0.7812 elapsed 130s
step 400: val loss 0.6116 val acc 0.8109 elapsed 172s  <- best
step 500: val loss 0.5479 val acc 0.8016 elapsed 214s
step 600: val loss 0.5471 val acc 0.8000 elapsed 256s
step 700: val loss 0.5795 val acc 0.8016 elapsed 298s
step 800: val loss 0.5534 val acc 0.8375 elapsed 341s  <- best
step 900: val loss 0.5143 val acc 0.8484 elapsed 381s  <- best
step 1000: val loss 0.5318 val acc 0.8422 elapsed 422s
loaded best checkpoint, val acc 0.8484


## 7. Try it on your own text

In [9]:
@torch.no_grad()
def predict(text):
    model.eval()
    ids, mask = encode(text)
    idx = torch.tensor([ids])
    m = torch.tensor([mask])
    logits, _ = model(idx, m)
    probs = torch.softmax(logits, dim=-1)[0]
    label = 'positive' if probs[1] > probs[0] else 'negative'
    return label, probs[1].item()

for text in [
    "This movie was a complete waste of time, terrible acting and a boring plot.",
    "An absolute masterpiece, beautifully shot with a gripping story from start to finish.",
    "It was okay, not great but not terrible either."
]:
    label, p_pos = predict(text)
    print(f"[{label:>8}] (p_positive={p_pos:.2f})  {text}")

[negative] (p_positive=0.01)  This movie was a complete waste of time, terrible acting and a boring plot.
[positive] (p_positive=0.99)  An absolute masterpiece, beautifully shot with a gripping story from start to finish.
[negative] (p_positive=0.01)  It was okay, not great but not terrible either.
